# Hypergraph

From the corpus to the hypergraph of the report's Section 3, in its order: the grid, the nodes, the hyperedges, their weights, and the choice of shapes. Reads `data/interim/00_pieces.csv`, `01_corpus.csv`, `02_corpus.csv` and the raw scores; writes the three panels of the report's Figure 1, `tex/figures/hyperedge_shapes_<a, b, c>.pdf`, and the hypergraph of each corpus, `data/interim/04_hypergraph_<corpus>.npz`.

- **Draft:** every section runs except the representation tests, which state what they will compute.

In [ ]:
import os, shutil, subprocess, tempfile
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
from math import comb
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from matplotlib.colors import to_rgb
from music21 import converter, expressions
from IPython.display import Image, display
from figures import palette   # the Nord colours of the house style

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
FIGURES = Path("../tex/figures")   # read by the report
WORKERS = os.cpu_count() // 2
BEAT = {"bach": 1, "palestrina": 2}   # the beat in quarter notes: a quarter in 4/4, a half note in 4/2
FOLDS = 10   # as in 02_corpus
ORDER = ["Soprano", "Alto", "Tenor", "Baritone", "Bass"]   # voice types, from high to low
NAME12 = ["C", "C#", "D", "Eb", "E", "F", "F#", "G", "Ab", "A", "Bb", "B"]

def types_of(lines): return [ORDER.index(line.split()[0]) for line in lines.split("|")]   # each voice's type, as its place in ORDER

corpus = pd.read_csv(INTERIM / "01_corpus.csv").set_index("file")   # one row per file
versions = pd.read_csv(INTERIM / "02_corpus.csv")   # one row per version of a file: what the model counts
pieces = pd.read_csv(INTERIM / "00_pieces.csv").set_index("file")   # the dump of 00_understanding: every part of every file
duration = pieces["duration"]   # in quarter notes, for a check
corpus["types"], versions["types"] = corpus["lines"].map(types_of), versions["lines"].map(types_of)
corpus["instruments"] = pieces.loc[corpus.index, "parts"] > corpus["voices"]   # parts left in 01_filtering: where they played alone, the voices are silent
assert set(versions["file"]) == set(corpus.index) and set(versions["fold"]) == set(range(FOLDS))

## Grid

The normalised corpus: only facts read from the score; intervals, durations, texture and beat strength are derived later, with the hyperedges.

- **Cell:** a quarter of the beat, a sixteenth in Bach and an eighth in Palestrina; it keeps the note sounding at its end (01_filtering). A grace note lasts no time and fills no cell.
- **Per voice and cell:** the pitch, moved by the file's shift, or a rest; and an attack flag: the voice attacks a new note in the cell, or holds the note of the cell before (DeepBach's hold symbol, BachBot's tie flag, Allan & Williams' continued and repeated notes). A rest attacks nothing.
- **Per cell:** the position in the bar (cell 0 to 15), whose use the walk decides by an ablation; and the fermata, when a note or rest sounding at the end of the cell carries one, in any voice. Palestrina's files carry no fermata, and their barlines are the editor's.
- **Per version, from 02_corpus:** the cells of the skipped spans leave, and so do the interludes of the chorales whose instruments left in 01_filtering; a segment number marks each run of consecutive cells, so that no window spans a gap.
- **Per file, already in 01 and 02:** voice types, Bach's mode, Palestrina's tonal type, weight, group, fold.
- **Left out:** When in Rome's Roman numerals (the walk knows no chords) and the sung text (no Palestrina file has one).

In [ ]:
REST = -1   # the pitch of a cell where the voice is silent
STEP = {name: beat / 4 for name, beat in BEAT.items()}   # the cell in quarter notes

def grid_of(score, parts, step):   # per voice and cell, the pitch sounding at the cell's end and whether it starts in the cell; per cell, the position in the bar and the fermata
    n = round(float(score.highestTime) / step)
    ends = (np.arange(n) + 1) * step - 1e-6   # just before the end of each cell
    pitch = np.full((len(parts), n), REST)
    attack = np.zeros((len(parts), n), dtype=bool)
    fermata = np.zeros(n, dtype=bool)
    for v, k in enumerate(parts):
        notes = [x for x in score.parts[k].stripTies().flatten().notes if not x.duration.isGrace]   # a tied note is one note
        start = np.array([float(x.offset) for x in notes])
        end = start + [float(x.quarterLength) for x in notes]
        assert (end[:-1] <= start[1:]).all()   # one note at a time
        i = np.searchsorted(start, ends) - 1   # the last note that starts before the cell ends
        sounding = (i >= 0) & (ends < end[i])
        pitch[v, sounding] = [notes[j].pitch.midi for j in i[sounding]]
        attack[v, sounding] = start[i[sounding]] > ends[sounding] - step   # the note starts inside the cell
        for x in score.parts[k].flatten().notesAndRests:
            if any(isinstance(e, expressions.Fermata) for e in x.expressions):
                fermata |= (float(x.offset) <= ends) & (ends < float(x.offset + x.quarterLength))
    measures = list(score.parts[parts[0]].getElementsByClass("Measure"))
    starts = np.arange(n) * step
    m = np.searchsorted([float(x.offset) for x in measures], starts, side="right") - 1
    into = starts - [float(measures[j].offset) - float(measures[j].paddingLeft) for j in m]   # music21 pads a pickup on the left, so its beats keep their numbers
    bar_cell = np.rint(into / step).astype(int) % 16   # % 16: bwv119.9 ends on a breve in a bar of 4/4, read as two tied whole notes
    return {"pitch": pitch, "attack": attack, "bar_cell": bar_cell, "fermata": fermata}

In [ ]:
# a toy score, computed by hand: a pickup of one beat (cells 0-3 sit at 12-15 of the bar); a tied note, one attack (cell 4);
# an ornament, whose first thirty-second gives way to the second in one cell (19); a rest; a fermata (24-27); a repeated note,
# two attacks (28, 30); a grace note, no cell
TOY = """**kern
*M4/4
4c
=1
2d[
4d]
8.e
32f
32g
=2
4r
4g;
8g
8g
8aq
4a
==
*-
"""
toy_score = grid_of(converter.parse(TOY, format="humdrum"), [0], STEP["bach"])
C4, D4, E4, G4, A4 = 60, 62, 64, 67, 69
assert toy_score["pitch"][0].tolist() == [C4] * 4 + [D4] * 12 + [E4] * 3 + [G4] + [REST] * 4 + [G4] * 8 + [A4] * 4
assert np.flatnonzero(toy_score["attack"][0]).tolist() == [0, 4, 16, 19, 24, 28, 30, 32]
assert toy_score["bar_cell"].tolist() == [12, 13, 14, 15] + list(range(16)) * 2
assert np.flatnonzero(toy_score["fermata"]).tolist() == [24, 25, 26, 27]
print("Toy score: as computed by hand.")

In [ ]:
def file_of(args):   # the grid of a whole file, transposed, and the counts it is checked against
    path, parts, shift = args
    score = converter.parse(path)
    step = STEP[path.parent.name]
    grid = grid_of(score, parts, step)
    grid["pitch"] = np.where(grid["pitch"] == REST, REST, grid["pitch"] + shift)
    notes = [x for k in parts for x in score.parts[k].flatten().notes]   # as written: a tied note in pieces
    on_grid = [x for x in notes if not x.duration.isGrace and np.isclose(float(x.offset) / step, round(float(x.offset) / step))]
    counts = {"started": sum(x.tie is None or x.tie.type == "start" for x in notes),   # 03_description's count of the notes a voice starts
              "grace": sum(x.duration.isGrace for x in notes),
              "off grid": sum(not x.duration.isGrace for x in notes) - len(on_grid),
              "beat disagrees": sum((float(x.beat) - 1) * 4 != grid["bar_cell"][round(float(x.offset) / step)] for x in on_grid)}   # four cells per beat in both corpora
    return grid, counts

files = corpus.index
with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    out = list(pool.map(file_of, [(RAW / c / f, [int(k) for k in p.split("|")], int(s)) for f, c, p, s in
                                  zip(files, corpus["corpus"], corpus["voice_parts"], corpus["shift"])], chunksize=4))
full = {f: grid for f, (grid, _) in zip(files, out)}   # one grid per file; the versions are cut from these below
checks = pd.DataFrame([counts for _, counts in out], index=files)
checks["cells"] = [len(g["bar_cell"]) for g in full.values()]
checks["attacks"] = [int(g["attack"].sum()) for g in full.values()]
checks.groupby(corpus["corpus"])[["cells", "started", "attacks", "off grid", "grace"]].sum()

### Checks

- **Toy score:** computed by hand, above.
- **Length:** cells times the step equals the file's duration in 00_understanding.
- **Attacks:** every note that 03_description counts starts one cell, except the notes off the grid (the ornaments of 01_filtering) and the grace notes.
- **Position in the bar:** music21's beat at every note on the grid.
- **Held notes:** a held cell keeps the pitch of the cell before.
- **Transposition:** the lowest note of the last chord is the final of 01_filtering moved by the shift.

In [ ]:
assert (checks["cells"] * corpus["corpus"].map(STEP) == duration[files]).all()
assert (checks["attacks"] == checks["started"] - checks["off grid"] - checks["grace"]).all()
assert (checks["beat disagrees"] == 0).all()
for g in full.values():
    held = ~g["attack"] & (g["pitch"] != REST)
    assert not held[:, 0].any() and (g["pitch"][:, 1:][held[:, 1:]] == g["pitch"][:, :-1][held[:, 1:]]).all()
print("Off the grid:", ", ".join(f"{f} ({n})" for f, n in checks["off grid"].items() if n),
      "| grace notes:", ", ".join(f"{f} ({n})" for f, n in checks["grace"].items() if n))

def lowest_last(g):   # the lowest note of the last cell where a voice sounds
    last = g["pitch"][:, (g["pitch"] != REST).any(axis=0)][:, -1]
    return last[last != REST].min()
final = pd.Series({f: NAME12[lowest_last(g) % 12] for f, g in full.items()})
assert (final == [NAME12[(NAME12.index(x) + int(s)) % 12] for x, s in zip(corpus["final"], corpus["shift"])]).all()
pd.crosstab([corpus["corpus"], corpus["key"].str.split().str[1].fillna("")], final.rename("final after the shift"))

**Result**
- Every check holds. Bach: of 82,122 notes started, 82,112 start a cell; the other ten are the nine ornament notes off the grid, in five chorales, and one grace note (`bwv315`, bar 11). Palestrina: 632,389 notes, every one starts a cell.
- After the shift, 171 of 183 major chorales end on C and 135 of 169 minor ones on A; 25 minor ones end on E, the dominant.
- Palestrina's finals after the shift: D 326, G 296, C 240, A 202, E 106, F 6, B 3.

### Versions

- **Rule:** each row of 02_corpus takes its file's grid without the cells of its skipped spans.
- **Interludes:** in the chorales whose instrumental parts left in 01_filtering, the cells where every voice is silent leave too, and the sequence breaks there: the instruments played alone, so the silence is not the voices' music (our rule; DeepBach leaves such chorales out). Elsewhere a rest of every voice stays, as written.
- **Check:** the cells left are the file's cells less the length of the spans and, in those chorales, less the cells where every voice is silent outside the spans.

In [ ]:
def version_of(grid, skip, step, interludes):   # the cells of the skipped spans leave, and the interludes if asked; `segment` numbers each run of consecutive cells
    keep = np.ones(len(grid["bar_cell"]), dtype=bool)
    for span in ([] if pd.isna(skip) else skip.split(";")):
        start, end = (float(x) / step for x in span.split("-"))
        assert start.is_integer() and end.is_integer()   # a span starts and ends on the grid
        keep[int(start):int(end)] = False
    if interludes:   # the instruments played alone where every voice is silent
        keep &= (grid["pitch"] != REST).any(axis=0)
    segment = np.concatenate([[0], np.cumsum(np.diff(np.flatnonzero(keep)) > 1)])
    return {k: v[..., keep] for k, v in grid.items()} | {"segment": segment}

grids = [version_of(full[f], skip, STEP[c], corpus.at[f, "instruments"]) for f, skip, c in zip(versions["file"], versions["skip"], versions["corpus"])]   # aligned with the rows of `versions`
spanned = [version_of(full[f], skip, STEP[c], False) for f, skip, c in zip(versions["file"], versions["skip"], versions["corpus"])]   # the spans alone, for the checks
versions["cells"] = [len(g["segment"]) for g in grids]
versions["segments"] = [g["segment"][-1] + 1 for g in grids]
spans = versions["skip"].fillna("").str.findall(r"([\d.]+)-([\d.]+)").map(lambda s: sum(float(e) - float(b) for b, e in s))
assert ([len(g["segment"]) for g in spanned] == checks.loc[versions["file"], "cells"].values - spans / versions["corpus"].map(STEP)).all()   # spans never overlap
versions["interlude cells"] = [int((g["pitch"] == REST).all(axis=0).sum()) * corpus.at[f, "instruments"] for g, f in zip(spanned, versions["file"])]
assert (versions["cells"] == [len(g["segment"]) for g in spanned] - versions["interlude cells"]).all()
print(f"Chorales whose instruments left: {corpus['instruments'].sum()}, {(versions.groupby('file')['interlude cells'].first() > 0).sum()} of them with interludes.")
versions.groupby("corpus").agg(rows=("file", "size"), files=("file", "nunique"), cells=("cells", "sum"), interlude_cells=("interlude cells", "sum"),
                               rows_with_a_break=("segments", lambda s: (s > 1).sum()))

**Result:** Bach, 359 rows and 73,576 cells, 41 rows with a break; the interludes take 1,896 cells (counted once per version), in 10 of the 31 chorales whose instruments left. Palestrina, 1,181 rows and 911,664 cells, 4 rows with a break.

### Fermatas

- **Where:** per Bach chorale, the share of cells under a fermata; the chorales without one.

In [ ]:
bach = corpus.index[corpus["corpus"] == "bach"]
under = pd.Series({f: full[f]["fermata"].mean() for f in bach})
assert not any(full[f]["fermata"].any() for f in corpus.index[corpus["corpus"] == "palestrina"])
print(f"Cells under a fermata: {100 * sum(full[f]['fermata'].sum() for f in bach) / checks.loc[bach, 'cells'].sum():.1f}% of Bach's cells; "
      f"chorales without a fermata: {', '.join(under.index[under == 0])}.")

**Result:** 18.6% of Bach's cells lie under a fermata. Four chorales carry none (`bwv1.6`, `bwv112.5`, `bwv248.64-6`, `bwv79.3`), so the fermata marks phrase ends only where a file has them.

## Nodes

- **Node:** a voice type with a value, a transposed pitch or a rest (report, Section 3.1); Tenor 1 and Tenor 2 share the nodes of their type. Each corpus has its own nodes; a node's id is its row in the table.
- **Measure:** per corpus and voice type, the pitches sung in the counted corpus (the rows of 02_corpus), the lowest and the highest, and the semitones between them never sung.

In [ ]:
def name_of(midi): return f"{NAME12[midi % 12]}{midi // 12 - 1}"

sung = defaultdict(set)   # (corpus, voice type) -> the pitches it sings in the counted corpus
for name, types, g in zip(versions["corpus"], versions["types"], grids):
    for t, row in zip(types, g["pitch"]):
        sung[name, t] |= set(row[row != REST].tolist())
nodes = {name: pd.DataFrame([(t, p) for (c, t), pitches in sorted(sung.items()) if c == name for p in [REST, *sorted(pitches)]], columns=["type", "pitch"])
         for name in ["bach", "palestrina"]}   # per corpus, a node for each voice type and value, a rest included
print("Nodes, a rest for each voice type included:", {name: len(n) for name, n in nodes.items()})
pd.DataFrame([{"corpus": c, "voice": ORDER[t], "pitches": len(p), "lowest": name_of(min(p)), "highest": name_of(max(p)), "never sung": max(p) - min(p) + 1 - len(p)}
              for (c, t), p in sorted(sung.items())]).set_index(["corpus", "voice"])

**Result**
- Bach: 131 nodes, 127 pitches and a rest for each of the four voices. Palestrina: 135, 130 pitches and a rest for each of the five types.
- The ranges after the shift reach past the usual ones: Bach's soprano goes down to E3 in `bwv254`, a chorale low in all four voices (soprano A3 to B♭4 as written), moved five semitones down, from D minor to A minor, by the nearest shift.
- Few semitones inside a range are never sung: at most 4 in Bach, 6 in Palestrina's baritone, the rarest type.

### Voices that cross

- **Why:** the voice stays in the node because voices cross (report, Section 3.1).
- **Measure:** the cells where two voices of different types both sing, and the share of them where the higher type sings below the lower one. Two voices of one type (Tenor 1, Tenor 2) share their nodes and are left out.

In [ ]:
both, crossed, pairs = Counter(), Counter(), Counter()
for f, g in full.items():
    name, types = corpus.at[f, "corpus"], corpus.at[f, "types"]
    sounding = g["pitch"] != REST
    two, cross = np.zeros(len(g["bar_cell"]), dtype=bool), np.zeros(len(g["bar_cell"]), dtype=bool)
    for hi in range(len(types)):
        for lo in range(len(types)):
            if types[hi] < types[lo]:
                pair = sounding[hi] & sounding[lo]
                below = pair & (g["pitch"][hi] < g["pitch"][lo])
                two, cross = two | pair, cross | below
                pairs[name, f"{ORDER[types[hi]]} below {ORDER[types[lo]]}"] += int(below.sum())
    both[name] += int(two.sum())
    crossed[name] += int(cross.sum())
display(pd.DataFrame({"cells with two types": both, "crossed %": {k: round(100 * crossed[k] / both[k], 1) for k in both}}))
pd.Series(pairs).rename("cells").sort_values(ascending=False).groupby(level=0).head(3).to_frame()

**Result**
- Bach: in 2.4% of the cells where two voice types sing, the higher type sings below the lower one, most often the alto below the tenor.
- Palestrina: 14.8%, most often the alto below the tenor (95,374 cells). The types come from the instrument codes of the kern files (01_filtering).

## Hyperedges

A hyperedge is a brick with a counter and a ticket: the brick is the pitches, the counter is how often the corpus uses it, and the ticket holds the details, each with its own counts.

- **Windows:** every shape v × s, with v from 1 to 4 and s from 1 to 4, or up to 16 (a bar) for one voice and up to 8 (two beats) for two voices, whose cross-entropy still fell at four cells (section Shapes by cross-entropy), slides over every start cell and every set of v voices of the file (in Palestrina, any v of its voices): s consecutive cells of the grid, the minimal unit.
- **Class, the brick:** the scale degree of the lowest note of the first cell (its interval modulo 12 from C: Witten & Conklin's interval from the referent) and the interval of every voice in every cell from it, a rest included. Below three voices the class also names the voice types, since the line and the counterpoint belong to named parts. From three voices up the hyperedge is a sonority, and the same chord sung by other parts is the same class.
- **Ticket:** the attack pattern (which voice attacks a new note in which cell), the parts that sing the window, the position in the bar of its first cell and, in Bach, the fermata of each cell (in up to four cells the pattern is none, all, or a fermata that starts or ends inside the window). Each detail is decided in one place: held or sung again on the ticket of the line, a property of the voice's step (the decision of 2026-10-21); the parts and the fermata, both given by the piece, on the ticket of the sonority; the position in the bar for an ablation.
- **First cell silent:** the referent is the lowest note of the first cell where a voice sounds (our rule); a window where no voice sounds has no degree.
- **Voices:** in the order of the parts. Below three voices two parts of one type (Tenor 1, Tenor 2) keep that order, so one pair sung the other way round is another class.
- **From a class to its hyperedges:** a class placed on a referent pitch and on the parts of its ticket gives the node of each voice in each cell, one hyperedge for each octave of the referent.
- **Modes:** major and minor counted together; apart, as Hadjeres et al. 2016 and Allan & Williams 2004 do, as an ablation.
- **Exact cells:** the cells of every set of v voices are also counted with their exact pitches, the base level of the cross-entropy below.
- **Rule:** no window spans a skipped span.
- **Stored, per corpus:** the nodes; per shape, the classes and the windows counted by class, fold and ticket; per number of voices, the exact cells counted by fold.
- **Checks:** a toy computed by hand, with a chord sung by two sets of parts; the number of windows per version, C(V, v) × Σ (segment length − s + 1) for V voices; every window of three versions per corpus decoded back to its nodes and its attacks; the commonest classes of four voices in one cell.

In [ ]:
LONGEST = {1: 16, 2: 8, 3: 4, 4: 4}   # cells of the longest shape measured: a bar for one voice, two beats for two voices, one beat for three and four
SHAPES = [(v, s) for v in range(1, 5) for s in range(1, LONGEST[v] + 1)]   # v voices over s cells
HARMONY = 3   # from three voices up a hyperedge is a sonority, and the parts that sing it go on its ticket
# the bytes of a class: its degree, each voice's type (below three voices), then for each voice and cell its interval from the referent plus OFF
OFF, SILENT, NONE = 64, 127, 12   # an interval is stored plus 64, so that it is positive; 127 is a rest; 12 is the degree of a window without a note
EXACT_REST = 255   # an exact cell stores the pitch; 255 is a rest
TYPES = len(ORDER)   # the parts of a window as one number, one digit per voice in base 5

def void(key): return np.ascontiguousarray(key).view(np.dtype((np.void, key.shape[1]))).ravel()   # one comparable value per row

def starts_of(segment, s):   # the first cells of the windows of s cells that lie inside one segment
    return np.flatnonzero(segment[: len(segment) - s + 1] == segment[s - 1:])

def windows_of(g, types, v, s):   # per window of shape v × s in one version: its class, the pitches; and its ticket: attacks, parts, position in the bar, fermata
    starts = starts_of(g["segment"], s)
    sets = np.array(list(combinations(range(len(types)), v)))   # every set of v voices, in the order of the parts
    def cut(x): return sliding_window_view(x, s, axis=1)[sets][:, :, starts].transpose(0, 2, 1, 3).reshape(-1, v, s)   # (windows, v, s)
    pitch, attack = cut(g["pitch"]), cut(g["attack"])
    sounding = pitch != REST
    low = np.where(sounding, pitch, 999).min(axis=1)   # the lowest note of each cell
    referent = low[np.arange(len(low)), (low < 999).argmax(axis=1)]   # in the first cell where a voice sounds
    interval = np.where(sounding, pitch - referent[:, None, None] + OFF, SILENT)
    assert ((interval >= 0) & (interval <= SILENT)).all()
    degree = np.where(low.min(axis=1) < 999, referent % 12, NONE)
    voices = np.repeat(np.array(types)[sets], len(starts), axis=0)   # (windows, v)
    head = [degree[:, None]] if v >= HARMONY else [degree[:, None], voices]
    key = np.concatenate(head + [interval.reshape(len(pitch), -1)], axis=1).astype(np.uint8)
    ticket = {"attacks": (attack.reshape(len(pitch), -1).astype(np.int64) << np.arange(v * s)).sum(axis=1),   # bit s·i + k: voice i attacks in cell k
              "voices": (voices * TYPES ** np.arange(v)).sum(axis=1),
              "bar_cell": np.tile(g["bar_cell"][starts], len(sets)),
              "fermata": np.tile((sliding_window_view(g["fermata"], s)[starts].astype(np.int64) << np.arange(s)).sum(axis=1), len(sets))}   # bit k: cell k under a fermata
    return key, ticket

def exact_of(g, sets, types):   # per set of voices and cell, the voice types and each voice's exact pitch: (sets, cells, 2v)
    exact = np.where(g["pitch"][sets] == REST, EXACT_REST, g["pitch"][sets]).transpose(0, 2, 1)
    return np.concatenate([np.broadcast_to(np.array(types)[sets][:, None, :], exact.shape), exact], axis=2).astype(np.uint8)

def nodes_of(keys, v, s, referents, voices, nodes):   # classes placed on their referents and on the parts of their tickets: per window, the node of each voice in each cell
    cells = keys[:, (1 if v >= HARMONY else 1 + v):].reshape(len(keys), v, s).astype(int)
    assert ((keys[:, 0] == NONE) | (keys[:, 0] == referents % 12)).all()
    pitch = np.where(cells == SILENT, REST, referents[:, None, None] + cells - OFF)
    types = np.broadcast_to(((voices[:, None] // TYPES ** np.arange(v)) % TYPES)[:, :, None], pitch.shape)
    return pd.MultiIndex.from_frame(nodes).get_indexer(pd.MultiIndex.from_arrays([types.ravel(), pitch.ravel()])).reshape(pitch.shape)   # -1: a pitch the voice type never sings

def described(key, v, s):   # a class in words: the degree, then per voice its intervals from the referent; voices named below three, numbered from the top above
    names = [str(i + 1) for i in range(v)] if v >= HARMONY else [ORDER[t][:3] for t in key[1:1 + v]]
    def cell(c): return "rest" if c == SILENT else f"{int(c) - OFF:+d}"
    return " | ".join(["no note" if key[0] == NONE else NAME12[key[0]]] +
                      [f"{name}: " + " ".join(cell(c) for c in row) for name, row in zip(names, key[(1 if v >= HARMONY else 1 + v):].reshape(v, s))])

# a toy, computed by hand: the soprano rests, then attacks G4 and holds it; the alto sings C4, D4, D4 again; a fermata on the last cell
toy_grid = {"pitch": np.array([[REST, 67, 67], [60, 62, 62]]), "attack": np.array([[0, 1, 0], [1, 1, 1]], dtype=bool),
            "segment": np.zeros(3, dtype=int), "bar_cell": np.arange(3), "fermata": np.array([0, 0, 1], dtype=bool)}
key, ticket = windows_of(toy_grid, [0, 1], 1, 2)
assert [described(k, 1, 2) for k in key] == ["G | Sop: rest +0", "G | Sop: +0 +0", "C | Alt: +0 +2", "D | Alt: +0 +0"]
assert ticket["attacks"].tolist() == [2, 1, 3, 3]   # G4 held: attacked in the first cell only; D4 sung again: attacked in both
key, ticket = windows_of(toy_grid, [0, 1], 2, 2)
assert [described(k, 2, 2) for k in key] == ["C | Sop: rest +7 | Alt: +0 +2", "D | Sop: +5 +5 | Alt: +0 +0"]
assert ticket["attacks"].tolist() == [14, 13] and ticket["fermata"].tolist() == [0, 2]
toy_nodes = pd.DataFrame({"type": [0, 0, 1, 1, 1], "pitch": [REST, 67, REST, 60, 62]})   # soprano rest, G4; alto rest, C4, D4
assert nodes_of(key, 2, 2, np.array([60, 62]), ticket["voices"], toy_nodes).tolist() == [[[0, 1], [3, 4]], [[1, 1], [4, 4]]]
# a sonority: soprano and alto both on G4, tenor C4, bass C3; sung by soprano, tenor, bass or by alto, tenor, bass it is one class with two tickets
chord = {"pitch": np.array([[67], [67], [60], [48]]), "attack": np.ones((4, 1), dtype=bool), "segment": np.zeros(1, dtype=int),
         "bar_cell": np.zeros(1, dtype=int), "fermata": np.zeros(1, dtype=bool)}
key, ticket = windows_of(chord, [0, 1, 2, 4], 3, 1)
assert [described(k, 3, 1) for k in key] == ["C | 1: +7 | 2: +7 | 3: +0", "C | 1: +19 | 2: +19 | 3: +0", "C | 1: +19 | 2: +12 | 3: +0", "C | 1: +19 | 2: +12 | 3: +0"]
assert ticket["voices"].tolist() == [55, 105, 110, 111]   # Sop-Alt-Ten, Sop-Alt-Bas, Sop-Ten-Bas, Alt-Ten-Bas
print("Toy windows: as computed by hand.")

In [ ]:
h = {}   # the hypergraph of each corpus: its nodes; per shape, its classes and its windows counted by class and ticket; per number of voices, its exact cells
TICKET = ["attacks", "voices", "bar_cell", "fermata"]
for name in ["bach", "palestrina"]:
    rows = versions.index[versions["corpus"] == name]
    h[name] = {"nodes": nodes[name], "shapes": {}, "exact": {}}
    for v, s in SHAPES:
        parts = [(r, *windows_of(grids[r], versions.at[r, "types"], v, s)) for r in rows if len(versions.at[r, "types"]) >= v]   # (row, classes, ticket)
        key = np.concatenate([k for _, k, _ in parts])
        assert len(key) == sum(comb(len(versions.at[r, "types"]), v) * np.maximum(np.bincount(grids[r]["segment"]) - s + 1, 0).sum() for r in rows)
        classes, cls = np.unique(void(key), return_inverse=True)
        windows = pd.DataFrame({"class": cls.ravel().astype(np.int32),
                                "fold": np.concatenate([np.full(len(k), versions.at[r, "fold"], dtype=np.int8) for r, k, _ in parts]),
                                "weight": np.concatenate([np.full(len(k), versions.at[r, "weight"]) for r, k, _ in parts])}
                               | {column: np.concatenate([t[column] for _, _, t in parts]).astype(np.int32) for column in TICKET})
        h[name]["shapes"][v, s] = {"classes": classes.view(np.uint8).reshape(len(classes), -1),
                                   "counts": windows.groupby(["class", "fold", *TICKET]).agg(windows=("weight", "size"), weight=("weight", "sum")).reset_index()}
    for v in range(1, 5):   # the exact cells of every set of v voices
        mine = [r for r in rows if len(versions.at[r, "types"]) >= v]
        exact = [exact_of(grids[r], np.array(list(combinations(range(len(versions.at[r, "types"])), v))), versions.at[r, "types"]).reshape(-1, 2 * v) for r in mine]
        classes, cls = np.unique(void(np.concatenate(exact)), return_inverse=True)
        windows = pd.DataFrame({"class": cls.ravel().astype(np.int32),
                                "fold": np.concatenate([np.full(len(e), versions.at[r, "fold"], dtype=np.int8) for r, e in zip(mine, exact)]),
                                "weight": np.concatenate([np.full(len(e), versions.at[r, "weight"]) for r, e in zip(mine, exact)])})
        h[name]["exact"][v] = {"classes": classes.view(np.uint8).reshape(len(classes), -1),
                               "counts": windows.groupby(["class", "fold"]).agg(windows=("weight", "size"), weight=("weight", "sum")).reset_index()}

rng = np.random.default_rng(0)   # check: every window of three versions per corpus decodes back to its nodes, and its ticket back to its attacks
for name in h:
    for r in rng.choice(versions.index[versions["corpus"] == name], 3, replace=False):
        g, types = grids[r], versions.at[r, "types"]
        for v, s in SHAPES:
            if len(types) >= v:
                sets, starts = np.array(list(combinations(range(len(types)), v))), starts_of(g["segment"], s)
                def cut(x): return sliding_window_view(x, s, axis=1)[sets][:, :, starts].transpose(0, 2, 1, 3).reshape(-1, v, s)
                pitch = cut(g["pitch"])
                low = np.where(pitch != REST, pitch, 999).min(axis=1)
                key, ticket = windows_of(g, types, v, s)
                ids = nodes_of(key, v, s, low[np.arange(len(low)), (low < 999).argmax(axis=1)], ticket["voices"], h[name]["nodes"])
                attacks = ((ticket["attacks"][:, None] >> np.arange(v * s)) & 1).reshape(-1, v, s).astype(bool)
                assert (ids >= 0).all() and (h[name]["nodes"]["pitch"].to_numpy()[ids] == pitch).all() and (attacks == cut(g["attack"])).all()
print("Every window of three versions per corpus decodes back to its nodes and its attacks.")
pd.DataFrame([{"corpus": name, "shape": f"{v} × {s}", "windows": int(x["counts"]["windows"].sum()), "classes": len(x["classes"])}
              for name in h for (v, s), x in h[name]["shapes"].items()]).pivot(index="shape", columns="corpus").loc[[f"{v} × {s}" for v, s in SHAPES]]

In [ ]:
for name in ["bach", "palestrina"]:   # the commonest classes of four voices in one cell, as a share of the weighted windows
    x = h[name]["shapes"][4, 1]
    weight = x["counts"].groupby("class")["weight"].sum()
    print(name.capitalize(), *(f"  {100 * w / weight.sum():.2f}%  {described(x['classes'][k], 4, 1)}" for k, w in weight.nlargest(3).items()), sep="\n")

**Result**
- Every count of windows matches its formula, and every window of the six versions checked decodes back to its nodes and its attacks. Per shape, Bach has 72,310 to 441,456 windows, Palestrina 4.3 to 9.5 million.
- The classes grow with the shape: Bach 52 of one voice in one cell, 149,318 of one voice over sixteen cells, 227,118 of two voices over eight, 36,026 of four voices over four; Palestrina 64, 439,876, 1,623,644 and 1,755,893.
- Bach's commonest sonorities are the C major triad, root doubled at two octaves (3.19% of the windows of four voices in one cell), the same triad in close position (2.24%) and the G major triad (1.81%). Palestrina's commonest is four voices resting (1.07%), written rests since its files have no instruments; then one voice on D (0.61%) or on G (0.60%) above three resting voices: with five to eight voices, a set of four often holds a single voice that sings.

### Figure 1: from the score to the hyperedges

- **Why:** Figure 1 of the report, three panels that the report sets on one row, each with its own caption under one caption of the figure: (a) the bars as written; (b) the same bars on the grid, with the borders of the cells, where a note that fills two cells becomes two tied notes, attacked then held; (c) hyperedges on the grid, each a coloured blob around the notes of v voices over s cells, with its shape above it.
- **Excerpt:** the first two bars of BWV 40.6, transposed as the model reads them: four voices in quarter notes, then a half note; one accidental, the bass's F♯. Not one of the three Bach files under the CCARH copyright.
- **Four hyperedges, one of each shape of the base model and one of 4 × 2, in Nord colours:** 4 × 2 on cells 1–2, 2 × 2 (soprano, alto) on cells 3–4, then in bar 2 4 × 1 on cell 5 and 1 × 2 (soprano) on the half note, cells 7–8; no two overlap.
- **Drawing:** one PDF per panel, `tex/figures/hyperedge_shapes_<a, b, c>.pdf`, all of one width and one border, so that at one scale their staves line up. MusiXTeX engraves each panel on four staves; TikZ draws the cell borders and the hyperedges behind the notes, at positions that `tikzmark` saves on the noteheads, so LaTeX runs twice.
- **Simplification:** one cell per beat for readability; the model's cell is a quarter of a beat, so a quarter note fills four cells.

In [ ]:
FILE, BARS = "bwv40.6.mxl", 2
row = corpus.loc[FILE]
score = converter.parse(RAW / "bach" / FILE)
parts = [int(k) for k in row["voice_parts"].split("|")]
top = score.parts[parts[0]]
start = float(top.measure(1).getOffsetBySite(top))
notes = [[n for n in score.parts[k].stripTies().flatten().notes if start <= float(n.offset) < start + 4 * BARS] for k in parts]
assert all([float(n.quarterLength) for n in v] == [1, 1, 1, 1, 1, 1, 2] for v in notes)   # six quarter notes, then a half note
HALF = 6   # the index of the half note, on beats 3-4 of bar 2
pitches = [[n.pitch.midi + int(row["shift"]) for n in v] for v in notes]   # [voice][note], transposed
pitches

In [ ]:
DIATONIC = {0: (0, ""), 1: (0, "sh"), 2: (1, ""), 3: (1, "sh"), 4: (2, ""), 5: (3, ""), 6: (3, "sh"),
        7: (4, ""), 8: (4, "sh"), 9: (5, ""), 10: (5, "sh"), 11: (6, "")}   # pitch class to diatonic step and accidental, black keys as sharps
BOTTOM = [30, 30, 23, 18]   # diatonic number of each staff's bottom line: E4 for soprano and alto, E3 for the tenor's octave treble, G2 for the bass
CELLS = [0, 1, 2, 3, 4, 5, HALF, HALF]   # the note in each cell, one cell per beat: the half note fills two
CEILING, FLOOR = 11, -4   # the staff positions where the cell borders end, above the soprano and below the bass

def position(v, midi):   # MusiXTeX's note position, 0 on the bottom line, and the accidental
    step, accidental = DIATONIC[midi % 12]
    return (midi // 12 - 1) * 7 + step - BOTTOM[v], accidental

nord = palette["categorical"]
HYPEREDGES = [("1\\,\\texttimes\\,2", nord[5], [(0, 6), (0, 7)]),   # cells: (voice, cell)
              ("2\\,\\texttimes\\,2", nord[1], [(0, 2), (1, 2), (0, 3), (1, 3)]),
              ("4\\,\\texttimes\\,1", nord[4], [(v, 4) for v in range(4)]),
              ("4\\,\\texttimes\\,2", nord[0], [(v, c) for v in range(4) for c in (0, 1)])]

def excerpt(panel):   # the two bars on four staves, as written (a) or one note per cell (b, c); a tikzmark at the centre of every notehead
    lines = [r"{\smallmusicsize\elemskip=2.6pt", r"\begin{music}", r"\parindent0mm\nobarnumbers\interstaff{9}", r"\instrumentnumber{4}",
             r"\setstaffs11\setclef1{6}", r"\setstaffs21\setclef2{0}\setclefsymbol2{\treblelowoct}", r"\setstaffs31\setclef3{0}", r"\setstaffs41\setclef4{0}",
             r"\generalsignature{0}\generalmeter{\meterfrac44}", r"\startextract"]
    notes = list(range(HALF + 1)) if panel == "a" else CELLS
    for c, b in enumerate(notes):
        fields = []
        for v in (3, 2, 1, 0):   # MusiXTeX counts instruments from the bottom staff
            p, accidental = position(v, pitches[v][b])
            held = c > 0 and notes[c - 1] == b
            field = r"\zcharnote{%d}{\kern0.6\Interligne\tikzmark{%s-%d-%d}}" % (p, panel, v, c)
            if c == 0 and v in (0, 3):   # where the cell borders end
                field += r"\zcharnote{%d}{\tikzmark{%s-%s}}" % ((CEILING, panel, "ceiling") if v == 0 else (FLOOR, panel, "floor"))
            if accidental and not held:
                field += r"\%s{%d}" % (accidental, p)
            if c + 1 < len(notes) and notes[c + 1] == b:   # a tie to the next cell, away from the stem
                field += r"\itie%s{%d}{%d}" % ("d" if p < 4 else "u", v, p)
            if held:
                field += r"\ttie{%d}" % v
            fields.append(field + r"\%s{%d}" % (("h" if panel == "a" and b == HALF else "q") + ("u" if p < 4 else "l"), p))   # stem up below the middle line
        wide = (panel == "a" and b == HALF) or (c + 1 < len(notes) and any(position(v, pitches[v][notes[c + 1]])[1] for v in range(4)))   # room for a half note, or before an accidental
        lines.append((r"\NOTes " if wide else r"\NOtes ") + "&".join(fields) + r"\en" + (r"\bar" if c == 3 else ""))
    return "\n".join(lines + [r"\zendextract", r"\end{music}}"]) + "\n"

def borders(panel):   # the cells' inner borders, halfway between noteheads, from above the soprano to below the bass; the barline marks the border of the bars
    ends = rf"\coordinate ({panel}ceiling) at (pic cs:{panel}-ceiling); \coordinate ({panel}floor) at (pic cs:{panel}-floor);"
    return ends + " ".join(rf"\coordinate (m) at ($(pic cs:{panel}-0-{c - 1})!0.5!(pic cs:{panel}-0-{c})$); \draw[cell] (m |- {panel}ceiling) -- (m |- {panel}floor);"
                           for c in range(1, len(CELLS)) if c != 4)

def hull(k, cells):   # the window's corners, clockwise from the top left, filled and stroked 9 pt wide so the blob reaches past the noteheads; its shape above the cell borders
    voices, columns = sorted({v for v, _ in cells}), sorted({c for _, c in cells})
    corners = [(voices[0], columns[0]), (voices[0], columns[-1]), (voices[-1], columns[-1]), (voices[-1], columns[0])]
    path = " -- ".join(f"(pic cs:c-{v}-{c})" for v, c in dict.fromkeys(corners))
    label = rf"\coordinate (m) at ($(pic cs:c-0-{columns[0]})!0.5!(pic cs:c-0-{columns[-1]})$); \node[anchor=base, font=\sffamily\footnotesize] at ($(m |- cceiling) + (0, 3pt)$) {{{HYPEREDGES[k][0]}}};"
    return rf"\fill[h{k}] {path} -- cycle; \draw[h{k}, line width=9pt, line join=round, line cap=round] {path} -- cycle; " + label

shades = [r"\definecolor{h%d}{rgb}{%.4f,%.4f,%.4f}" % (k, *(0.45 * np.array(to_rgb(c)) + 0.55)) for k, (_, c, _) in enumerate(HYPEREDGES)]   # lightened on white, opaque
head = [r"\documentclass[border={2pt 8pt 2pt 12pt}]{standalone}",   # left, bottom, right, top: the same in every panel, so that their staves line up; room for what the overlay draws outside the box
        r"\usepackage{musixtex}", r"\usepackage{tikz}", r"\usetikzlibrary{tikzmark,calc}", *shades,
        r"\definecolor{grid}{HTML}{%s}\tikzset{cell/.style={draw=grid, line width=0.6pt}}" % palette["roles"]["grid"][1:]]
overlay = {"a": "", "b": borders("b"), "c": borders("c") + " " + " ".join(hull(k, cells) for k, (_, _, cells) in enumerate(HYPEREDGES))}

with tempfile.TemporaryDirectory() as tmp:
    FIGURES.mkdir(parents=True, exist_ok=True)
    for name in "abc":   # one PDF per panel: the report sets them side by side, each with its own caption
        Path(tmp, f"excerpt_{name}.tex").write_text(excerpt(name))   # MusiXTeX needs its own file: inside an argument, & would be read as a column
        doc = head + [r"\begin{document}", r"\tikz[remember picture,overlay]{" + overlay[name] + "}%", rf"\parbox{{4.6cm}}{{\input{{excerpt_{name}.tex}}}}", r"\end{document}"]   # the overlay before the music, so behind the notes; one width for the three panels, a little more than the music of the widest
        Path(tmp, f"{name}.tex").write_text("\n".join(doc) + "\n")
        run = subprocess.run(["latexmk", "-lualatex", "-interaction=nonstopmode", f"{name}.tex"], cwd=tmp, capture_output=True, text=True)
        assert run.returncode == 0, run.stdout[-2000:]   # fail loud, with LaTeX's own message
        shutil.copy(Path(tmp, f"{name}.pdf"), FIGURES / f"hyperedge_shapes_{name}.pdf")
        subprocess.run(["pdftoppm", "-png", "-r", "150", "-singlefile", f"{name}.pdf", name], cwd=tmp, check=True)   # a PNG to show here: Jupyter does not draw a PDF
        display(Image(filename=str(Path(tmp, f"{name}.png"))))
print("Written:", *(FIGURES / f"hyperedge_shapes_{name}.pdf" for name in "abc"))

## Weights

- **Rule:** the support of a class is the share of the windows of its shape that belong to it, each window counted 1/k times for a file with k versions (Agrawal et al. 1993 for the term).
- **Tickets:** the probability of a detail given the class is its count with the class, plus the escape of method C (Moffat 1988, in Witten & Bell 1991) to the detail's share in the whole shape: a class with n counts and r distinct details leaves r / (n + r) to that share, so a class seen rarely, and always the same way, still leaves room for the detail it never showed. A class never counted takes the share alone.
- **Fermata (Bach):** no support of its own, since the melody brings it; on the sonority's ticket the classes are counted apart for each fermata pattern.
- **When:** computed from the counts when needed, on the complete corpus or without one fold; not stored.
- **Checks:** the ten folds make the complete corpus; the support without a fold equals the complete counts less that fold, normalised; Bach's windows of s cells, up to four, fall into 2s fermata patterns; for 200 classes, the ticket of the attacks sums to 1 over every pattern of the shape.

In [ ]:
def by_fold(x):   # the weight of each class in each fold: (classes, folds)
    w = np.zeros((len(x["classes"]), FOLDS))
    np.add.at(w, (x["counts"]["class"].to_numpy(), x["counts"]["fold"].to_numpy()), x["counts"]["weight"].to_numpy())
    return w

def support(x, without=None, fermata=None):   # the support of each class: on the complete corpus, or without one fold; in Bach, among the windows of one fermata pattern
    c = x["counts"]
    if without is not None:
        c = c[c["fold"] != without]
    if fermata is not None:
        c = c[c["fermata"] == fermata]
    w = np.bincount(c["class"], c["weight"], minlength=len(x["classes"]))
    return w / w.sum()

def ticket(x, column, without=None):   # P(value of an annotation | class): a value unseen with its class takes the escape of method C to its share of the shape
    c = x["counts"] if without is None else x["counts"][x["counts"]["fold"] != without]
    joint = c.groupby(["class", column])["weight"].sum()
    n, r = joint.groupby(level=0).sum(), joint.groupby(level=0).size()   # per class, its weight and its number of distinct values
    general = c.groupby(column)["weight"].sum() / c["weight"].sum()
    def probability(classes, values):
        seen = joint.reindex(pd.MultiIndex.from_arrays([classes, values])).fillna(0).to_numpy()
        n_c, r_c, g = n.reindex(classes).fillna(0).to_numpy(), r.reindex(classes).fillna(0).to_numpy(), general.reindex(values).fillna(0).to_numpy()
        return np.where(n_c > 0, (seen + r_c * g) / np.maximum(n_c + r_c, 1e-300), g)   # a class never counted: the shape's share alone
    return probability, general

for name in h:
    for (v, s), x in h[name]["shapes"].items():
        w = by_fold(x)
        complete = w.sum(axis=1)
        assert np.isclose(complete.sum(), x["counts"]["weight"].sum()) and np.allclose(support(x), complete / complete.sum())
        assert all(np.allclose(support(x, without=f), (complete - w[:, f]) / (complete.sum() - w[:, f].sum())) for f in (0, FOLDS - 1))
        if name == "bach" and s <= 4:   # a longer window can hold a whole fermata
            assert x["counts"]["fermata"].nunique() == 2 * s   # none, all, or a fermata that starts or ends at one of the s - 1 inner borders
    x = h[name]["shapes"][1, 2]   # check: on a sample of classes, the ticket of the attacks sums to 1 over every pattern of the shape
    probability, general = ticket(x, "attacks", without=0)
    sample = np.random.default_rng(0).choice(len(x["classes"]), 200, replace=False)
    assert np.allclose([probability(np.full(len(general), k), general.index.to_numpy()).sum() for k in sample], 1)
print("The supports agree with the counts in every shape, and the tickets sum to 1.")
patterns = h["bach"]["shapes"][1, 4]["counts"].groupby("fermata")["weight"].sum()
(100 * patterns / patterns.sum()).round(2).rename(lambda p: format(p, "04b")[::-1]).rename("% of Bach's windows of one voice over four cells").to_frame()

**Result**
- Every check holds in every shape, and the tickets sum to 1.
- A window of four cells has eight fermata patterns: of Bach's weighted windows, 73.14% have none, 10.57% lie all under a fermata, and each pattern where a fermata starts or ends inside the window holds 2.45% to 2.98%.

### The ticket of the line

- **Why:** the line decides whether a voice that keeps its pitch holds the note or sings it again.
- **Measure:** when a voice keeps its pitch from one cell to the next, the share of the cases where it sings the note again; and the held-out bits of the attack pattern of every window of one voice over two cells, given the shape's share of each pattern alone, or given the ticket of the class (ten folds).

In [ ]:
held = []   # the ticket of the line at work: when a voice keeps its pitch from one cell to the next, how often it sings the note again
for name in h:
    x = h[name]["shapes"][1, 2]
    same = (x["classes"][:, 2] == x["classes"][:, 3]) & (x["classes"][:, 2] != SILENT)   # classes where the voice keeps its pitch
    c = x["counts"][same[x["counts"]["class"].to_numpy()]]
    again = c["weight"][(c["attacks"] & 2) > 0].sum() / c["weight"].sum()   # bit 1: an attack in the second cell
    bits = {"the shape's share": 0.0, "the ticket of the class": 0.0}
    for f in range(FOLDS):   # held-out bits of the attack pattern of every window of a fold
        out = x["counts"][x["counts"]["fold"] == f]
        probability, general = ticket(x, "attacks", without=f)
        bits["the shape's share"] -= (out["weight"] * np.log2(general.reindex(out["attacks"]).to_numpy())).sum()
        bits["the ticket of the class"] -= (out["weight"] * np.log2(probability(out["class"].to_numpy(), out["attacks"].to_numpy()))).sum()
    held.append({"corpus": name, "kept pitch sung again %": 100 * again} | {f"bits per window, {k}": b / x["counts"]["weight"].sum() for k, b in bits.items()})
pd.DataFrame(held).set_index("corpus").round(3)

**Result:** a voice that keeps its pitch sings the note again in 4.9% of the cases in Bach and 2.3% in Palestrina; the rest are held notes. Knowing the class, the pitches, cuts the uncertainty of the attacks by 42% in Bach (from 1.59 to 0.93 bits per window) and by 47% in Palestrina (from 1.17 to 0.62), on held-out folds.

### Do the rules on copies and repeats bias the supports?

- **Why:** a composition counts once, however often it was printed, encoded or written out (01_filtering, Copies; 02_corpus, Repeats), while a formula that recurs across different pieces keeps its full weight. The check measures whether the rules move the supports that the walk reads.
- **Measure:** per shape, the share of the support that moves (total variation distance) between the corpus with the rules (versions weighted 1/k, a written-out repeat once) and without them (every file whole, weight 1; the interludes out in both); and the largest change among the twenty commonest classes, also among those with a note.
- **Yardstick:** the share that moves when one fold, a tenth of the music, is left out; the median over the ten folds.
- **Left out:** the 23 identical copies removed in 01_filtering, each the same file twice (in Palestrina, the same mass and page of the edition); this notebook does not read them.

In [ ]:
whole = {f: version_of(full[f], np.nan, STEP[corpus.at[f, "corpus"]], corpus.at[f, "instruments"]) for f in corpus.index}   # every file whole, every repeat in; the interludes out, as in the grids
bias = []
for name in h:
    files = [f for f in corpus.index[corpus["corpus"] == name]]
    for (v, s), x in h[name]["shapes"].items():
        plain, counts = np.unique(void(np.concatenate([windows_of(whole[f], corpus.at[f, "types"], v, s)[0] for f in files if len(corpus.at[f, "types"]) >= v])), return_counts=True)
        ruled = void(x["classes"])
        both, where = np.unique(np.concatenate([ruled, plain]), return_inverse=True)
        p = np.bincount(where[:len(ruled)], support(x), minlength=len(both))   # with the rules
        q = np.bincount(where[len(ruled):], counts / counts.sum(), minlength=len(both))   # without them
        w = by_fold(x)
        complete = w.sum(axis=1)
        folds = [0.5 * np.abs(complete / complete.sum() - (complete - w[:, f]) / (complete.sum() - w[:, f].sum())).sum() for f in range(FOLDS)]
        top = np.argsort(p)[::-1][:20]   # the twenty commonest classes with the rules
        sounding = top[both.view(np.uint8).reshape(len(both), -1)[top, 0] != NONE]   # those with a note
        bias.append({"corpus": name, "shape": f"{v} × {s}", "moved by the rules %": 100 * 0.5 * np.abs(p - q).sum(),
                     "moved by one fold %": 100 * np.median(folds), "top 20, largest change %": 100 * np.abs(q[top] / p[top] - 1).max(),
                     "top 20 with a note, largest change %": 100 * np.abs(q[sounding] / p[sounding] - 1).max()})
bias = pd.DataFrame(bias)
bias["ratio"] = bias["moved by the rules %"] / bias["moved by one fold %"]
print(f"The rules move {bias['moved by the rules %'].min():.2f}% to {bias['moved by the rules %'].max():.2f}% of a shape's support; one fold left out moves "
      f"{bias['moved by one fold %'].min():.2f}% to {bias['moved by one fold %'].max():.2f}%. Among the twenty commonest classes of a shape, the largest change is "
      f"{bias['top 20, largest change %'].max():.1f}%, and {bias['top 20 with a note, largest change %'].max():.1f}% among those with a note.")
bias.set_index(["corpus", "shape"]).unstack(0).round(2).loc[[f"{v} × {s}" for v, s in SHAPES]]

**Result**
- Palestrina: the rules move at most 0.14% of a shape's support, under a tenth of what leaving out one fold moves.
- Bach: they move from 0.49% (one voice, one cell) to 6.46% (one voice, sixteen cells), between 0.69 and 0.76 times what leaving out one fold moves.
- Among the twenty commonest classes of a shape, the largest change is 16.1%, a class with notes. With the interludes out (section Versions), the silence of every voice no longer moves; before, it gained 54% without the rules.
- So the rules move the supports less than leaving out a tenth of the music does.

## Shapes by cross-entropy

- **Why:** the literature chooses orders and viewpoints by cross-entropy on held-out music (Witten & Conklin 1993; Whorley et al. 2013); no source sets a threshold on coverage (`bib/coverage_threshold_2026-10-27.md`).
- **Measure:** for each shape v × s, the bits per voice per cell of the pitches of the last cell of its v voices given the s − 1 cells before it (the attacks are on the ticket); counts on nine folds, test on the tenth, each version with its weight. The shapes v × 1 give the exact cell with no context.
- **Targets:** every cell whose cells before it, as many as the longest shape of v voices needs (fifteen for one voice, seven for two, three for three and four), lie in its segment, and whose cell before has a note in one of the v voices. So every context has a referent, the class of a window fixes the pitches of its last cell, and all shapes of v voices predict the same cells.
- **Unseen classes:** the escape of method C (Moffat 1988, in Witten & Bell 1991): a context with n counts and r distinct continuations leaves r / (n + r) to the context one cell shorter, which spreads it over every continuation (our interpolation). Below one cell of context, the exact pitches given the voice types; below them, every voice's pitches and the rest, all equally likely. No fitted weight.
- **Choice:** per number of voices, the number of cells with the fewest bits. Comparing numbers of voices as models of the whole texture needs the walk's rule for combining hyperedges, still open.
- **Coverage, as a description:** the share of the targets whose window class occurs in the nine folds (Sears et al. 2017 measure the same on chord progressions).
- **Checks:** every held-out window is a class of the complete corpus; for one voice, the probabilities of every next cell that an interval can reach sum to 1.

In [ ]:
def targets_of(g, types, v):   # the targets of one version: per order, the key of the window that ends on each target; and each target's set of voices
    sets = np.array(list(combinations(range(len(types)), v)))
    longest = LONGEST[v]
    t = starts_of(g["segment"], longest) + longest - 1   # cells with longest - 1 cells before them in their segment
    keys = {1: exact_of(g, sets, types)[:, t]}
    for s in range(2, longest + 1):
        first = starts_of(g["segment"], s)
        at = np.full(len(g["segment"]), -1)
        at[first] = np.arange(len(first))
        keys[s] = windows_of(g, types, v, s)[0].reshape(len(sets), len(first), -1)[:, at[t - s + 1]]
    heard = (g["pitch"][sets][:, :, t - 1] != REST).any(axis=1)   # a note in the cell before: (sets, cells)
    return {s: k[heard] for s, k in keys.items()}, np.repeat(np.arange(len(sets)), len(t))[heard.ravel()]

def escaped(n_event, n_context, distinct, lower):   # method C: a context with n counts and r distinct continuations leaves r / (n + r) to the order below
    return np.where(n_context > 0, (n_event + distinct * lower) / np.maximum(n_context + distinct, 1e-300), lower)

In [ ]:
xent, per_fold = [], []
for name in h:
    rows = versions.index[versions["corpus"] == name]
    values = {t: len(p) + 1 for (c, t), p in sung.items() if c == name}   # one voice: each pitch it sings, or a rest
    for v in range(1, 5):
        mine = [r for r in rows if len(versions.at[r, "types"]) >= v]
        x = h[name]["exact"][v]
        counted = {1: (void(x["classes"]), by_fold(x), np.unique(void(x["classes"][:, :v]), return_inverse=True)[1].ravel())}   # per number of cells: classes, weights by fold, context of each class; here the voice types
        for s in range(2, LONGEST[v] + 1):
            x = h[name]["shapes"][v, s]
            head = 1 if v >= HARMONY else 1 + v   # degree, and the voice types below three voices
            before = x["classes"][:, head:].reshape(len(x["classes"]), v, s)[:, :, :s - 1].reshape(len(x["classes"]), -1)
            counted[s] = void(x["classes"]), by_fold(x), np.unique(void(np.concatenate([x["classes"][:, :head], before], axis=1)), return_inverse=True)[1].ravel()   # context: the first s - 1 cells
        ids, fold, weight, uniform = {s: [] for s in counted}, [], [], []
        for r in mine:
            types = versions.at[r, "types"]
            keys, which = targets_of(grids[r], types, v)
            for s, k in keys.items():
                i = np.searchsorted(counted[s][0], void(k))
                assert (counted[s][0][i] == void(k)).all()   # every held-out window is a class of the complete corpus
                ids[s].append(i)
            fold.append(np.full(len(which), versions.at[r, "fold"]))
            weight.append(np.full(len(which), versions.at[r, "weight"]))
            uniform.append(1 / np.prod([[values[types[k]] for k in voice_set] for voice_set in combinations(range(len(types)), v)], axis=1)[which])
        ids = {s: np.concatenate(i) for s, i in ids.items()}
        fold, weight, uniform = np.concatenate(fold), np.concatenate(weight), np.concatenate(uniform)
        bits, seen = {s: np.zeros(len(fold)) for s in counted}, {s: np.zeros(len(fold), dtype=bool) for s in counted}
        for f in range(FOLDS):
            out = fold == f
            p = uniform[out]
            for s in range(1, LONGEST[v] + 1):   # from the exact cell up to the longest context
                _, w, context = counted[s]
                n = w.sum(axis=1) - w[:, f]
                c = ids[s][out]
                p = escaped(n[c], np.bincount(context, n)[context[c]], np.bincount(context, n > 0)[context[c]], p)
                bits[s][out], seen[s][out] = -np.log2(p) / v, n[c] > 0
        for s in counted:
            xent.append({"corpus": name, "voices": v, "cells": s, "bits": np.average(bits[s], weights=weight),
                         "seen %": 100 * np.average(seen[s], weights=weight), "targets": len(fold)})
            per_fold += [{"corpus": name, "voices": v, "cells": s, "fold": f, "bits": np.average(bits[s][fold == f], weights=weight[fold == f])} for f in range(FOLDS)]
xent, per_fold = pd.DataFrame(xent), pd.DataFrame(per_fold)
by_fold_bits = per_fold[per_fold["cells"] > 1].pivot_table(index=["corpus", "voices", "fold"], columns="cells", values="bits")
chosen = xent[xent["cells"] > 1].loc[lambda d: d.groupby(["corpus", "voices"])["bits"].idxmin()].set_index(["corpus", "voices"])["cells"]
winners = by_fold_bits.idxmin(axis=1)   # per fold, the number of cells with the fewest bits
best = pd.DataFrame({"cells chosen": chosen, "folds where it wins": [(winners.loc[k] == c).sum() for k, c in chosen.items()]})
display(xent.pivot_table(index=["corpus", "voices"], columns="cells", values="bits").round(3).rename(columns=lambda s: f"{s} cell" + "s" * (s > 1)))
display(xent.pivot_table(index=["corpus", "voices"], columns="cells", values="seen %").round(1).rename(columns=lambda s: f"seen %, {s} cell" + "s" * (s > 1)))
best

In [ ]:
# check: one voice of Bach, one cell of context, fold 0 held out; over every next cell an interval can reach, the probabilities sum to 1
held_out = 0
x, exact = h["bach"]["shapes"][1, 2], h["bach"]["exact"][1]
w = by_fold(x)
n = w.sum(axis=1) - w[:, held_out]
count, n_context, distinct = dict(zip(map(bytes, x["classes"]), n)), defaultdict(float), defaultdict(int)
for k, c in zip(x["classes"], n):
    n_context[bytes(k[:3])] += c
    distinct[bytes(k[:3])] += c > 0
w = by_fold(exact)
exact_n = {bytes(k): c for k, c in zip(exact["classes"], w.sum(axis=1) - w[:, held_out]) if c > 0}   # (voice type, exact pitch) -> weight on the nine folds
rng = np.random.default_rng(0)
sums, unsung = [], []
for r in rng.choice(versions.index[(versions["corpus"] == "bach") & (versions["fold"] == held_out)], 5, replace=False):
    g = grids[r]
    for voice, kind in enumerate(versions.at[r, "types"]):
        t = rng.choice(np.flatnonzero((g["pitch"][voice, :-1] != REST) & (g["segment"][:-1] == g["segment"][1:]))) + 1
        ref = int(g["pitch"][voice, t - 1])
        context = bytes([ref % 12, kind, OFF])
        typed = {k: c for k, c in exact_n.items() if k[0] == kind}
        pitches = sung["bach", kind]
        total = outside = 0
        for pitch in [None, *range(ref - OFF, ref + SILENT - OFF)]:
            cell = SILENT if pitch is None else pitch - ref + OFF
            known = pitch is None or pitch in pitches
            p0 = (typed.get(bytes([kind, EXACT_REST if pitch is None else pitch]), 0) + len(typed) / (len(pitches) + 1)) / (sum(typed.values()) + len(typed)) if known else 0
            p1 = (count.get(context + bytes([cell]), 0) + distinct[context] * p0) / (n_context[context] + distinct[context]) if n_context[context] > 0 else p0
            total, outside = total + p1, outside + (0 if known else p1)
        sums.append(total)
        unsung.append(outside)
assert np.allclose(sums, 1)
print(f"Sums over {len(sums)} contexts: all 1 (largest error {max(abs(np.array(sums) - 1)):.1e}). "
      f"Probability on pitches the voice type never sings: at most {100 * max(unsung):.2f}%, mean {100 * np.mean(unsung):.2f}%.")

**Result**
- The context does most of the work: one cell before the target takes one voice from 3.89 to 1.38 bits per cell in Bach, and from 3.93 to 1.03 in Palestrina.
- The numbers of cells with the fewest bits: for one voice, 6 in Bach (1.068 bits) and 10 in Palestrina (0.676), in all ten folds; for two voices, 4 in Bach (1.041, in 8 of 10) and 6 in Palestrina (0.674, in all ten); for three voices, 4 in both (1.023 and 0.673, in all ten); for four voices, 4 in Bach (1.260, level with 3 cells to three decimals, in 6 of 10) and 2 in Palestrina (0.828, in 8 of 10).
- Past its best length each curve rises, as the classes get rare: at 8 cells half of Bach's targets of two voices (49.6%) have a class seen in the other nine folds, at 4 cells 53.1% of Bach's targets of four voices.
- Palestrina's notes are about twice as long as Bach's (0.59 notes per voice per beat against 1.10, medians of 03_description), so its best single line of 10 cells and Bach's of 6 both hold about one and a half notes.
- Check: over 20 contexts the probabilities sum to 1 within 4.4e-16. The classes carry no octave, so some probability falls on pitches the voice type never sings: at most 0.37%, 0.03% on average.

## The model's shapes

Fixed on 2026-10-29: a simple model of three families, and five shapes added only to test, one at a time, whether the style needs them. The moment is the cell, so lengths count quarters of a beat.

| Shape | Role | In the music | In the literature |
|---|---|---|---|
| 1 × 2 | base | the smallest step of a voice: it holds or moves, and by how much | pairs of notes of one voice (Hadjeres et al. 2016) |
| 2 × 2 | base | the motion of two voices from one cell to the next: parallel fifths and octaves | Fux's rules of motion (Fux 1725); evaluated, not modelled, by Hadjeres et al. |
| 4 × 1 | base | the sonority; 3 × 1 in Palestrina's three-voice movements | the chord as a hidden state (Allan & Williams 2004); a slice of four parts (Whorley et al. 2013) |
| 1 × 4 | ablation | one beat of a voice | melodic n-grams (Witten & Conklin 1993) |
| 1 × 8 | ablation | half a bar of a voice | melodic n-grams of a higher order (Witten & Conklin) |
| 2 × 4 | ablation | one beat of two voices, also when they do not move together | none in the sources we checked |
| 4 × 2 | ablation | the change of sonority | chord transitions (Allan & Williams); each voice against the previous slice (Conklin 2002) |
| 4 × 4 | ablation | one beat of four voices, with the passing notes inside it | n-grams of slices (Whorley et al.) |

- **Why 1 × 8 and no other eight-cell shape:** in the section above, one voice still predicts better at eight cells than at four (Bach 1.129 against 1.221 bits, Palestrina 0.783 against 0.943). Two voices predict worse at eight cells in Bach (1.176 against 1.041) but better in Palestrina (0.715 against 0.745, best at six cells, 0.674); four voices gain nothing from three to four cells in Bach and lose after two in Palestrina. We keep 2 × 4 in both corpora, so that the two styles have the same shapes.
- **Caveat:** the literature counts notes, slices or beats; we count cells, so one beat of a Bach line holds about one note.
- **Stored, per corpus:** the nodes; for each shape, its classes and its windows counted by class, fold and ticket (attacks, parts, position in the bar, fermata); the exact cells of one, two, three and four voices, the base of the escape. The complete corpus is every fold; the ten training corpora are the complete one less one fold each (`support(x, without=f)`).
- **Check:** the files read back equal to the arrays written.

In [ ]:
BASE, ADDED = [(1, 2), (2, 2), (4, 1)], [(1, 4), (1, 8), (2, 4), (4, 2), (4, 4)]   # added only for the ablation
saved = []
for name in h:
    shapes = BASE + ADDED + ([(3, 1)] if name == "palestrina" else [])   # 3 × 1: the sonority of the three-voice movements
    arrays = {"nodes": h[name]["nodes"].to_numpy()}
    for v, s in shapes:
        x = h[name]["shapes"][v, s]
        arrays[f"shape_{v}x{s}_classes"] = x["classes"]
        arrays |= {f"shape_{v}x{s}_{column}": x["counts"][column].to_numpy() for column in x["counts"]}
        saved.append({"corpus": name, "shape": f"{v} × {s}", "role": "base" if (v, s) in BASE or (v, s) == (3, 1) else "ablation",
                      "classes": len(x["classes"]), "windows": int(x["counts"]["windows"].sum())})
    for v in range(1, 5):
        x = h[name]["exact"][v]
        arrays[f"exact_{v}_classes"] = x["classes"]
        arrays |= {f"exact_{v}_{column}": x["counts"][column].to_numpy() for column in x["counts"]}
    path = INTERIM / f"04_hypergraph_{name}.npz"
    np.savez_compressed(path, **arrays)
    back = np.load(path)
    assert set(back.files) == set(arrays) and all(np.array_equal(back[k], a) for k, a in arrays.items())
    print(f"{path.name}: {path.stat().st_size / 1e6:.1f} MB, read back equal.")
pd.DataFrame(saved).pivot(index=["role", "shape"], columns="corpus").fillna(0).astype(int)

**Result:** the model's hypergraph has eight shapes, and in Palestrina 3 × 1 as well. In the base model Bach has 24,450 classes (780 of the line, 19,707 of the counterpoint, 3,963 sonorities) and Palestrina 144,641 (737, 79,237, and 56,297 sonorities of four voices with 8,370 of three); the shapes for the ablation go up to 77,048 classes in Bach (2 × 4) and 1,755,893 in Palestrina (4 × 4). The files take 3.2 MB (Bach) and 57.2 MB (Palestrina) and read back equal.

## Representation tests (to build)

- **Why:** the class keeps the degree of the global key; a test says what the degree adds and what the global key costs. A first version ran in a script on 2026-10-24, on a provisional grid; its numbers enter the report only from here.
- **Next note:** for every new note that follows a sounding note, the held-out bits per note (ten folds) of the interval alone, the global degree with the interval, the local degree with the interval (When in Rome's key at that point; Bach only, the 304 chorales with an aligned analysis), and the exact pitch.
- **Off the scale:** the share of probability each model puts on notes off the scale, against the real share; and the real share off the scale of the global and of the local key.
- **Key changes:** the share of chorales whose analysis changes key, and of their time outside the first key.
- **Literature, read in the PDFs:** no model normalises pitches to the local key; Hadjeres et al. 2016, BachBot and BacHMMachine transpose globally to C or A; DeepBach augments with every transposition and adds a key estimated bar by bar by music21 as metadata; BacHMMachine learns the local key as a hidden state from annotated chorales.